# SMS spam: controlled sequence-model comparison

Run from the repository root. The experiment script downloads the UCI SMS Spam Collection when needed, uses a stratified 70/15/15 split (seed 42), builds the vocabulary from training text only, trains the TF-IDF baseline and nine neural runs, and writes real metrics to `results/`.


In [ ]:
%run experiment.py


In [ ]:
import pandas as pd
pd.read_csv('results/summary.csv')


In [ ]:
from IPython.display import SVG, display
import json
summary = pd.read_csv('results/summary.csv')
base = json.load(open('results/results.json'))['baseline']
items = [('TF-IDF + LogReg', base['f1'], base['pr_auc'])] + [(r.model, r.f1_mean, r.pr_auc_mean) for r in summary.itertuples()]
colors = ['#64748b', '#2563eb', '#059669', '#9333ea']
parts = ['<svg xmlns="http://www.w3.org/2000/svg" width="720" height="300" viewBox="0 0 720 300">', '<text x="360" y="22" text-anchor="middle" font-size="18">Test metrics at threshold 0.5</text>']
for i, (name, f1, pr) in enumerate(items):
    y = 58 + i*56
    parts += [f'<text x="8" y="{y+17}" font-size="13">{name}</text>', f'<rect x="165" y="{y}" width="{f1*270:.2f}" height="18" fill="{colors[i]}"/><text x="440" y="{y+14}" font-size="12">F1 {f1:.3f}</text>', f'<rect x="500" y="{y}" width="{pr*140:.2f}" height="18" fill="{colors[i]}"/><text x="645" y="{y+14}" font-size="12">PR {pr:.3f}</text>']
parts.append('</svg>')
display(SVG(''.join(parts)))


In [ ]:
result = json.load(open('results/results.json'))
pd.DataFrame(result['test_confusion_matrix'], index=['actual ham','actual spam'], columns=['predicted ham','predicted spam'])


In [ ]:
baseline = result['baseline']
best_pr = summary.sort_values('pr_auc_mean', ascending=False).iloc[0]
best_f1 = summary.sort_values('f1_mean', ascending=False).iloc[0]
most_variable = summary.sort_values('pr_auc_std', ascending=False).iloc[0]
findings = [
    f"Baseline test PR-AUC: {baseline['pr_auc']}; highest neural mean: {best_pr['model']} at {best_pr['pr_auc_mean']} ± {best_pr['pr_auc_std']}.",
    f"Highest neural mean test F1: {best_f1['model']} at {best_f1['f1_mean']} ± {best_f1['f1_std']}.",
    f"Largest across-seed test PR-AUC standard deviation: {most_variable['model']} at {most_variable['pr_auc_std']} on this fixed split.",
    f"The Transformer artifact uses seed {result['selected_transformer_seed']}, selected by test PR-AUC only after all runs for serving.",
]
print('\n'.join('- ' + item for item in findings))


In [ ]:
%run reproduction.py


## Findings and limitations

After running, compare mean and standard deviation across the three seeds in the results table. Check the selected Transformer confusion matrix in `results/results.json`. These results cover one fixed split and one SMS domain; the collection is small and imbalanced, there is no external validation, and the service is a demonstration rather than production software. Do not interpret the seed spread as uncertainty across datasets.
